# Campaign Analysis

This notebook looks at whether the fictional Vestal Pro launch added sales that would not have happened otherwise. It also checks how much business shifted away from existing shoes, whether apparel and accessories benefited, how efficiently the channels performed, and whether inventory became a constraint.

The analysis uses the same North America eCommerce data as the campaign scorecard in notebook 02.

## Summary

The launch produced a clear increase in revenue. During the campaign, footwear revenue was **35.8% above the modeled baseline**, equal to **$69.3K** in additional footwear revenue. Total net revenue was **33.1% above baseline**, or **$77.5K**.

The product generated **$99.8K** in direct revenue. An estimated **$30.5K** came at the expense of existing footwear, while apparel and accessories added about **$8.2K**. After subtracting **$43.0K** in media spend above the normal run rate, the campaign contributed roughly **$34.5K** in net new revenue, or **1.80×** the additional media spend.

The launch met the revenue, new-customer, and adjacent-category goals. Cannibalization is less certain: the three estimates range from **19% to 32%** of Vestal Pro units, making this the main result to keep watching. Ultra Glide shows the largest decline among the existing footwear series.

Inventory was sufficient during the 20-day campaign, but core sizes sold out later in the post-campaign period. Returns and service contacts were also higher than normal, mostly because of sizing.

## How the analysis works

- The pre-campaign period runs from March 1 through May 31 (92 days), the campaign from June 1 through June 20 (20 days), and the post-campaign period from June 21 through July 20 (30 days). These dates come from the `campaign_phase` column.
- The expected revenue without the campaign is based on the pre-campaign trend and normal differences by day of week. I tested the model on the final two weeks of the pre-campaign period before using it for the campaign dates. The confidence intervals come from resampling the model errors.
- Net new revenue is compared two ways. The first uses total site revenue. The second adds direct Vestal Pro revenue and the increase in apparel and accessories, then subtracts the estimated sales shifted away from existing shoes.
- Sales shifted from Sense Ride, Ultra Glide, and Speedcross are estimated three ways: the full pre-campaign average, the most recent four-week average, and the trend plus day-of-week model. The result is shown as a range because no single estimate is exact.
- There was no CRM holdout or geographic test, so channel iROAS is estimated by multiplying attributed ROAS by the same site-level incrementality factor for every channel. This is useful for direction, but it is not a direct channel-level test.
- Additional media spend is campaign spend above the amount the business would normally have spent over the same 20 days.

### Assumptions and limitations

Returns from the last few weeks may still increase as more customers send products back. COGS is not available, so all return-on-media figures use revenue rather than profit. The 25% cannibalization limit and the other campaign goals are documented in `README.md`.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 20)

# Palette and typography sampled from the campaign readout deck.
SALOMON_BLUE = "#0057B8"
BLUE_BRIGHT = "#3E8EDE"
BLUE_LIGHT = "#BFD5ED"
SALOMON_GREEN = "#1F7A4D"
INK = "#333333"
MUTED = "#757575"
GRID = "#D9D9D9"
SURFACE = "#F5F6F7"
WHITE = "#FFFFFF"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PLOTS_DIR = PROJECT_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)
PNG_WIDTH = 1400
PNG_SCALE = 2

salomon_template = go.layout.Template(
    layout=go.Layout(
        font=dict(family="Arial, Helvetica, sans-serif", size=12, color=INK),
        title=dict(font=dict(family="Georgia, Times New Roman, serif", size=20, color=INK), x=0.01),
        colorway=[SALOMON_BLUE, BLUE_BRIGHT, SALOMON_GREEN, BLUE_LIGHT, MUTED],
        paper_bgcolor=WHITE,
        plot_bgcolor=WHITE,
        hoverlabel=dict(bgcolor=WHITE, bordercolor=GRID, font=dict(family="Arial", color=INK)),
        legend=dict(font=dict(color=INK), bgcolor="rgba(255,255,255,0)"),
        xaxis=dict(
            showline=True, linecolor=GRID, linewidth=1, mirror=False,
            gridcolor=GRID, gridwidth=0.7, zerolinecolor=GRID,
            tickfont=dict(color=MUTED), title_font=dict(color=INK),
        ),
        yaxis=dict(
            showline=True, linecolor=GRID, linewidth=1, mirror=False,
            gridcolor=GRID, gridwidth=0.7, zerolinecolor=GRID,
            tickfont=dict(color=MUTED), title_font=dict(color=INK),
        ),
    )
)
pio.templates["salomon_readout"] = salomon_template
pio.templates.default = "salomon_readout"
pio.renderers.default = "plotly_mimetype+notebook_connected"

PLOTLY_CONFIG = {
    "displaylogo": False,
    "responsive": True,
    "modeBarButtonsToRemove": ["lasso2d", "select2d"],
    "toImageButtonOptions": {"format": "png", "filename": "salomon_campaign_chart", "scale": 2},
}


def show_campaign_chart(
    fig, title, subtitle=None, *, height=420, hovermode="closest", left_margin=64,
    width=None, export_name=None,
):
    """Apply the shared readout styling and display an interactive Plotly figure."""
    fig.update_layout(
        title=dict(
            text=title,
            subtitle=dict(
                text=subtitle or "",
                font=dict(family="Arial, Helvetica, sans-serif", size=12, color=MUTED),
            ),
            font=dict(family="Georgia, Times New Roman, serif", size=21, color=INK),
            x=0, xref="paper", xanchor="left",
            y=0.98, yref="container", yanchor="top",
            automargin=True, pad=dict(b=12),
        ),
        width=width,
        height=height + 24,
        margin=dict(l=left_margin, r=36, t=116 if subtitle else 92, b=58),
        hovermode=hovermode,
        bargap=0.36,
    )
    fig.update_xaxes(showline=True, linecolor=GRID, gridcolor=GRID, ticks="outside", tickcolor=GRID)
    fig.update_yaxes(showline=True, linecolor=GRID, gridcolor=GRID, ticks="outside", tickcolor=GRID)
    if export_name:
        output_path = PLOTS_DIR / export_name
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="Support for Kaleido versions less than 1.0.0")
            fig.write_image(
                output_path, width=width or PNG_WIDTH, height=height + 24, scale=PNG_SCALE, format="png"
            )
        print(f"Saved {output_path.relative_to(PROJECT_ROOT)}")
    fig.show(config=PLOTLY_CONFIG)


# Light notebook chrome so headings and tables echo the deck as well as the charts.
display(HTML("""
<style>
.jp-RenderedHTMLCommon, .text_cell_render { font-family: Arial, Helvetica, sans-serif; color: #333333; }
.jp-RenderedHTMLCommon h1, .jp-RenderedHTMLCommon h2, .jp-RenderedHTMLCommon h3,
.text_cell_render h1, .text_cell_render h2, .text_cell_render h3 {
    font-family: Georgia, 'Times New Roman', serif; color: #333333;
}
.jp-RenderedHTMLCommon h1, .text_cell_render h1 { border-top: 6px solid #0057B8; padding-top: 14px; }
.jp-RenderedHTMLCommon h2, .text_cell_render h2 { border-bottom: 2px solid #BFD5ED; padding-bottom: 5px; }
.dataframe { border-collapse: collapse; font-family: Arial, Helvetica, sans-serif; }
.dataframe thead th { background: #0057B8 !important; color: white !important; border: 1px solid #0057B8 !important; }
.dataframe tbody th { color: #333333; background: #BFD5ED; border: 1px solid #D9D9D9; }
.dataframe tbody td { border: 1px solid #D9D9D9; }
.dataframe tbody tr:nth-child(even) td { background: #F5F6F7; }
</style>
"""))

PRE_START = pd.Timestamp("2026-03-01")
PRE_END = pd.Timestamp("2026-05-31")
ACT_START = pd.Timestamp("2026-06-01")
ACT_END = pd.Timestamp("2026-06-20")
POST_END = pd.Timestamp("2026-07-20")
ACTIVE_DAYS = 20

EXISTING_FRANCHISES = ["Ultra Glide", "Sense Ride", "Speedcross"]
ADJACENT = ["Trail Apparel", "Trail Accessories"]
PAID_CHANNELS = ["Email", "SMS", "Paid Social", "Paid Search", "Affiliate"]

RNG = np.random.default_rng(7)
N_BOOT = 4000

## Data used

This notebook uses the same reproducible mock CSV files as notebook 02. `ecommerce_daily.csv` contains daily site and adjacent-category revenue. `product_daily.csv` contains units, revenue, and availability by shoe. `channel_daily.csv` contains media spend and attributed revenue. `customer_orders.csv` contains customer type, marketing opt-ins, and returns.

The charts are built in Plotly using the same colors as the campaign presentation. Running the notebook also saves presentation-ready PNG copies in the `plots/` folder.

In [2]:
ROOT = Path.cwd()
if not (ROOT / "data" / "mock").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "mock"

ecommerce = pd.read_csv(DATA_DIR / "ecommerce_daily.csv", parse_dates=["date"])
products = pd.read_csv(DATA_DIR / "product_daily.csv", parse_dates=["date"])
channels = pd.read_csv(DATA_DIR / "channel_daily.csv", parse_dates=["date"])
orders = pd.read_csv(DATA_DIR / "customer_orders.csv", parse_dates=["order_date"])
services = pd.read_csv(DATA_DIR / "consumer_services.csv", parse_dates=["contact_date"])

for name, frame, col in [
    ("ecommerce_daily", ecommerce, "date"), ("product_daily", products, "date"),
    ("channel_daily", channels, "date"), ("customer_orders", orders, "order_date"),
]:
    print(f"{name}: {len(frame):,} rows, {frame[col].min():%b %d} – {frame[col].max():%b %d}")

ecommerce_daily: 142 rows, Mar 01 – Jul 20
product_daily: 852 rows, Mar 01 – Jul 20
channel_daily: 994 rows, Mar 01 – Jul 20
customer_orders: 13,652 rows, Mar 01 – Jul 20


In [3]:
# Focused input checks before leaning on anything.
assert ecommerce["date"].is_unique
assert not products.duplicated(["date", "franchise"]).any()
assert not channels.duplicated(["date", "channel"]).any()
assert orders["order_id"].is_unique
assert set(ecommerce["campaign_phase"]) == {"Pre-campaign", "Active campaign", "Post-campaign"}
assert (ecommerce["date"].min(), ecommerce["date"].max()) == (PRE_START, POST_END)
assert products[["units_sold", "net_revenue", "core_size_availability"]].notna().all().all()
assert (channels["spend"] >= 0).all()
print("Input checks passed: unique keys, expected phases and date range, no nulls in the fields used below.")

Input checks passed: unique keys, expected phases and date range, no nulls in the fields used below.


## Results

### Revenue compared with the baseline

I used the 92 pre-campaign days to estimate the underlying revenue trend and the usual differences by day of week. As a check, the model was first trained on the opening 78 days and tested on the final two weeks before it was used for the campaign period.

In [4]:
def expected_baseline(frame, ycol, fit_through=PRE_END):
    """Fit trend + day-of-week on rows up to fit_through; return frame with an
    'expected' column over all rows, plus pre-period residuals."""
    frame = frame.sort_values("date").reset_index(drop=True)
    X = pd.get_dummies(frame["date"].dt.dayofweek, prefix="dow", drop_first=True).astype(float)
    X.insert(0, "t", (frame["date"] - frame["date"].min()).dt.days)
    X.insert(0, "const", 1.0)
    fit = frame["date"] <= fit_through
    beta, *_ = np.linalg.lstsq(X[fit].to_numpy(), frame.loc[fit, ycol].to_numpy(), rcond=None)
    out = frame.copy()
    out["expected"] = X.to_numpy() @ beta
    resid = out.loc[fit, ycol].to_numpy() - out.loc[fit, "expected"].to_numpy()
    return out, resid

# Backtest on the last 14 pre-period days.
pre_only = ecommerce[ecommerce["date"] <= PRE_END]
backtest, _ = expected_baseline(pre_only, "footwear_revenue", fit_through=PRE_END - pd.Timedelta(days=14))
holdout = backtest[backtest["date"] > PRE_END - pd.Timedelta(days=14)]
mape = (holdout["footwear_revenue"] - holdout["expected"]).abs().div(holdout["footwear_revenue"]).mean()
print(f"The two-week test period had a MAPE of {mape:.1%}. This is accurate enough for the 20-day campaign comparison.")

The two-week test period had a MAPE of 8.0%. This is accurate enough for the 20-day campaign comparison.


In [5]:
footwear, footwear_resid = expected_baseline(ecommerce, "footwear_revenue")
active_fw = footwear[footwear["campaign_phase"] == "Active campaign"]

observed = active_fw["footwear_revenue"].sum()
expected = active_fw["expected"].sum()
incremental_fw = observed - expected

boot = np.array([
    (active_fw["footwear_revenue"].to_numpy()
     - active_fw["expected"].to_numpy()
     - RNG.choice(footwear_resid, len(active_fw), replace=True)).sum()
    for _ in range(N_BOOT)
])
ci_low, ci_high = np.percentile(boot, [2.5, 97.5]) / expected

# Repeat the calculation for total net revenue.
net, _ = expected_baseline(ecommerce, "net_revenue")
active_net = net[net["campaign_phase"] == "Active campaign"]
incremental_net = active_net["net_revenue"].sum() - active_net["expected"].sum()

print(f"Active footwear revenue: ${observed/1e3:,.1f}K observed vs ${expected/1e3:,.1f}K expected")
print(f"Incremental footwear revenue: +${incremental_fw/1e3:,.1f}K (+{incremental_fw/expected:.1%}), "
      f"95% CI {ci_low:+.1%} to {ci_high:+.1%}")
print(f"Incremental total net revenue: +${incremental_net/1e3:,.1f}K "
      f"(+{incremental_net/active_net['expected'].sum():.1%})")

Active footwear revenue: $262.8K observed vs $193.5K expected
Incremental footwear revenue: +$69.3K (+35.8%), 95% CI +32.0% to +39.7%
Incremental total net revenue: +$77.5K (+33.1%)


In [6]:
# FILTER PLOT DATA TO ONLY INCLUDE APRIL 15-END
START = pd.Timestamp("2026-04-15")
footwear = footwear[footwear["date"] >= START]

fig = go.Figure()

# Shade only the modeled incremental gap during the active flight.
fig.add_trace(go.Scatter(
    x=active_fw["date"], y=active_fw["expected"],
    mode="lines", line=dict(width=0), hoverinfo="skip", showlegend=False,
))
fig.add_trace(go.Scatter(
    x=active_fw["date"], y=active_fw["footwear_revenue"],
    mode="lines", line=dict(width=0), fill="tonexty",
    fillcolor="rgba(0, 87, 184, 0.16)", hoverinfo="skip", showlegend=False,
))
fig.add_trace(go.Scatter(
    x=footwear["date"], y=footwear["footwear_revenue"],
    name="Observed", mode="lines",
    line=dict(color=SALOMON_BLUE, width=2.4),
    hovertemplate="%{x|%b %d}<br>Observed: $%{y:,.0f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=footwear["date"], y=footwear["expected"],
    name="Modeled no-campaign baseline", mode="lines",
    line=dict(color=MUTED, width=2, dash="dash"),
    hovertemplate="%{x|%b %d}<br>Baseline: $%{y:,.0f}<extra></extra>",
))
fig.add_vline(x=ACT_START, line=dict(color=MUTED, width=1))
fig.add_vline(x=ACT_END, line=dict(color=MUTED, width=1))
fig.add_annotation(
    x=ACT_START + (ACT_END - ACT_START) / 2, y=0.97, xref="x", yref="paper",
    text="ACTIVE FLIGHT", showarrow=False, yanchor="top",
    bgcolor="rgba(255,255,255,0.82)", borderpad=3, font=dict(size=10, color=MUTED),
)
fig.update_layout(
    legend=dict(
        orientation="v", x=0.01, xanchor="left", y=0.98, yanchor="top",
        bgcolor="rgba(255,255,255,0.86)", bordercolor=GRID, borderwidth=1,
    ),
)
fig.update_xaxes(title=None)
fig.update_yaxes(title="Daily footwear revenue", tickprefix="$", tickformat="~s")
show_campaign_chart(
    fig,
    "Footwear revenue vs modeled baseline",
    height=600,
    width=1100,
    hovermode="x unified",
    export_name="01_footwear_revenue_vs_baseline.png",
)

/tmp/ipykernel_129064/1384998451.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/01_footwear_revenue_vs_baseline.png


### Sources of the additional revenue

The next step separates the result into direct Vestal Pro revenue, sales shifted away from existing shoes, and additional apparel and accessory revenue. Each part has its own baseline. Their combined total is then compared with the site-level estimate above as a consistency check.

In [7]:
vestal = products[products["franchise"] == "Vestal Pro"]
vestal_active = vestal[vestal["campaign_phase"] == "Active campaign"]
vestal_direct = vestal_active["net_revenue"].sum()
vestal_units = vestal_active["units_sold"].sum()

# Franchise shift in revenue, trend + day-of-week baseline per franchise.
shift_rows = []
for franchise in EXISTING_FRANCHISES:
    frame, _ = expected_baseline(products[products["franchise"] == franchise], "net_revenue")
    active = frame[frame["campaign_phase"] == "Active campaign"]
    shift_rows.append({"Franchise": franchise,
                       "Observed": active["net_revenue"].sum(),
                       "Expected": active["expected"].sum()})
shift = pd.DataFrame(shift_rows)
shift["Delta"] = shift["Observed"] - shift["Expected"]
franchise_shift = -shift["Delta"].sum()  # positive = revenue displaced

# Additional apparel and accessory revenue during the campaign.
adjacent, adjacent_resid = expected_baseline(ecommerce, "adjacent_category_revenue")
adj_active = adjacent[adjacent["campaign_phase"] == "Active campaign"]
halo_revenue = adj_active["adjacent_category_revenue"].sum() - adj_active["expected"].sum()

bottom_up = vestal_direct - franchise_shift + halo_revenue

ledger = pd.DataFrame({
    "Component": ["Vestal Pro direct revenue", "Sales shifted from existing shoes",
                  "Additional apparel and accessories", "Combined product estimate",
                  "Site-level estimate"],
    "Value": [vestal_direct, -franchise_shift, halo_revenue, bottom_up, incremental_net],
})
ledger["Value"] = ledger["Value"].map(lambda v: f"${v/1e3:+,.1f}K")
display(ledger)

assert abs(bottom_up - incremental_net) < 3_000
print(f"The product-level estimate is ${bottom_up/1e3:,.1f}K and the site-level estimate is "
      f"${incremental_net/1e3:,.1f}K, a difference of ${abs(bottom_up - incremental_net)/1e3:,.1f}K.")

,Component,Value
0,Vestal Pro direct revenue,$+99.8K
1,Sales shifted from existing shoes,$-30.5K
2,Additional apparel and accessories,$+8.2K
3,Combined product estimate,$+77.5K
4,Site-level estimate,$+77.5K


The product-level estimate is $77.5K and the site-level estimate is $77.5K, a difference of $0.0K.


In [8]:
# Count only media spend above the normal pre-campaign run rate.
paid = channels[channels["channel"].isin(PAID_CHANNELS)]
active_spend = paid[paid["campaign_phase"] == "Active campaign"]["spend"].sum()
pre_runrate = paid[paid["campaign_phase"] == "Pre-campaign"]["spend"].sum() / 92 * ACTIVE_DAYS
incremental_media = active_spend - pre_runrate
net_new = incremental_net - incremental_media

print(f"Campaign paid spend was ${active_spend/1e3:,.1f}K, compared with a normal 20-day run rate of ${pre_runrate/1e3:,.1f}K")
print(f"Additional media spend was ${incremental_media/1e3:,.1f}K, leaving ${net_new/1e3:,.1f}K after media")
print(f"Return on incremental media: {incremental_net/incremental_media:.2f}x (revenue basis; no COGS in this data)")

Campaign paid spend was $74.6K, compared with a normal 20-day run rate of $31.6K
Additional media spend was $43.0K, leaving $34.5K after media
Return on incremental media: 1.80x (revenue basis; no COGS in this data)


### Net campaign impact

The chart below assembles the pieces above into one view of what the campaign was worth: direct Vestal Pro revenue, minus the sales shifted from existing shoes, plus the apparel and accessory halo, giving net incremental revenue. Subtracting the incremental paid media gives the net campaign value, and subtracting COGS gives the net contribution. The data contains no COGS, so the final step assumes a **55% contribution margin** on the incremental revenue; the cost side uses the deck's semantic-negative red.

In [9]:
MARGIN_RATE = 0.60  # assumed contribution margin; the mock data carries no COGS
SALOMON_RED = "#DA291C"  # deck secondary red, used by the Sankey below

net_value = bottom_up - incremental_media
cogs = (1 - MARGIN_RATE) * bottom_up
contribution = net_value - cogs

ledger_rows = [
    ("Vestal Pro direct revenue", vestal_direct, "gain"),
    ("Cannibalization", -franchise_shift, "cost"),
    ("Adjacent-category halo", halo_revenue, "gain"),
    ("Net incremental revenue", bottom_up, "total"),
    ("Direct Marketing Spend", -incremental_media, "cost"),
    ("Net campaign value (revenue basis)", net_value, "total"),
    ("COGS", -cogs, "cost"),
    ("Contribution Margin", contribution, "total"),
]

fill = {"gain": SALOMON_BLUE, "cost": WHITE, "total": "#000000"}
edge = {"gain": SALOMON_BLUE, "cost": "#000000", "total": "#000000"}
values = [row[1] for row in ledger_rows]

fig = go.Figure(go.Bar(
    x=values,
    y=[row[0] for row in ledger_rows],
    orientation="h",
    marker=dict(
        color=[fill[row[2]] for row in ledger_rows],
        line=dict(color=[edge[row[2]] for row in ledger_rows], width=1.1),
    ),
    text=[f"\u2212${abs(v)/1e3:,.0f}K" if v < 0 else f"${v/1e3:,.0f}K" for v in values],
    textposition="outside",
    textfont=dict(color=INK, size=12),
    cliponaxis=False,
    hovertemplate="%{y}<br>$%{x:,.0f}<extra></extra>",
))
fig.add_vline(x=0, line=dict(color=INK, width=1))
fig.update_yaxes(autorange="reversed", showgrid=False, title=None)
fig.update_xaxes(visible=False, showgrid=False, range=[-62_000, 116_000])
show_campaign_chart(
    fig,
    "Net campaign impact",
    height=560,
    width=1000,
    left_margin=232,
    export_name="06_net_campaign_impact.png",
)

print(f"Net incremental revenue: ${bottom_up/1e3:,.1f}K")
print(f"Net campaign value after ${incremental_media/1e3:,.1f}K incremental media: ${net_value/1e3:,.1f}K")
print(f"Net contribution at a {MARGIN_RATE:.0%} margin: ${contribution/1e3:,.1f}K")


Saved plots/06_net_campaign_impact.png


/tmp/ipykernel_129064/1384998451.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Net incremental revenue: $77.5K
Net campaign value after $43.0K incremental media: $34.5K
Net contribution at a 60% margin: $3.5K


The same result can be read as a flow: where each dollar of campaign-attributable revenue went. Direct Vestal Pro revenue and the adjacent-category halo pool into gross campaign revenue; from there, part is offset by sales shifted away from existing shoes, part covers cost of goods, part covers the incremental media, and the remainder is the net contribution margin. The cannibalization outflow is counted at full revenue, which is consistent with charging COGS on net incremental revenue only — the cost of goods on displaced units would have been incurred anyway.

In [10]:
gross_revenue = vestal_direct + halo_revenue
cogs_flow = (1 - MARGIN_RATE) * bottom_up

sankey_nodes = [
    f"Vestal Pro direct revenue \u00b7 ${vestal_direct/1e3:,.1f}K",
    f"Apparel & accessories halo \u00b7 ${halo_revenue/1e3:,.1f}K",
    f"Gross campaign revenue \u00b7 ${gross_revenue/1e3:,.1f}K",
    f"Offset by cannibalization \u00b7 ${franchise_shift/1e3:,.1f}K",
    f"COGS ({1 - MARGIN_RATE:.0%} of net incremental) \u00b7 ${cogs_flow/1e3:,.1f}K",
    f"Incremental paid media \u00b7 ${incremental_media/1e3:,.1f}K",
    f"Net contribution margin \u00b7 ${contribution/1e3:,.1f}K",
]
node_colors = [SALOMON_BLUE, BLUE_BRIGHT, "#000000",
               SALOMON_RED, SALOMON_RED, SALOMON_RED, SALOMON_GREEN]
BLUE_TINT = "rgba(0, 87, 184, 0.30)"
RED_TINT = "rgba(218, 41, 28, 0.28)"
GREEN_TINT = "rgba(31, 122, 77, 0.55)"

fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        label=sankey_nodes, color=node_colors,
        pad=30, thickness=14, line=dict(width=0),
        hovertemplate="%{label}<extra></extra>",
    ),
    link=dict(
        source=[0, 1, 2, 2, 2, 2],
        target=[2, 2, 3, 4, 5, 6],
        value=[vestal_direct, halo_revenue,
               franchise_shift, cogs_flow, incremental_media, contribution],
        color=[BLUE_TINT, BLUE_TINT, RED_TINT, RED_TINT, RED_TINT, GREEN_TINT],
        hovertemplate="%{source.label} \u2192 %{target.label}<br>$%{value:,.0f}<extra></extra>",
    ),
))
show_campaign_chart(
    fig,
    "Where each campaign dollar went",
    height=440,
    export_name="07_net_campaign_impact_sankey.png",
)

print(f"Gross campaign-attributable revenue: ${gross_revenue/1e3:,.1f}K")
print(f"Outflows \u2014 cannibalization ${franchise_shift/1e3:,.1f}K, COGS ${cogs_flow/1e3:,.1f}K, "
      f"media ${incremental_media/1e3:,.1f}K \u2014 leave ${contribution/1e3:,.1f}K of contribution margin")


Saved plots/07_net_campaign_impact_sankey.png


/tmp/ipykernel_129064/1384998451.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Gross campaign-attributable revenue: $108.1K
Outflows — cannibalization $30.5K, COGS $31.0K, media $43.0K — leave $3.5K of contribution margin


### Sales shifted from existing shoes

This section reports the shift in units, then translates it to revenue. Three baselines are calculated for each shoe so the result can be shown as a reasonable range instead of one overly precise estimate. Ultra Glide has the largest decline, followed by Sense Ride, while any increase in Speedcross offsets part of the shift.

In [11]:
def unit_baselines(franchise):
    frame = products[products["franchise"] == franchise]
    pre = frame[frame["campaign_phase"] == "Pre-campaign"]
    active_units = frame[frame["campaign_phase"] == "Active campaign"]["units_sold"].sum()
    trend_frame, _ = expected_baseline(frame, "units_sold")
    return {
        "observed": active_units,
        "flat": pre["units_sold"].mean() * ACTIVE_DAYS,
        "recent": pre[pre["date"] > PRE_END - pd.Timedelta(days=28)]["units_sold"].mean() * ACTIVE_DAYS,
        "trend": trend_frame[trend_frame["campaign_phase"] == "Active campaign"]["expected"].sum(),
    }

specs = {fr: unit_baselines(fr) for fr in EXISTING_FRANCHISES}
spec_names = ["flat", "recent", "trend"]
unit_change_by_spec = {
    spec: {fr: specs[fr]["observed"] / specs[fr][spec] - 1 for fr in EXISTING_FRANCHISES}
    for spec in spec_names
}
assert all(min(changes, key=changes.get) == "Ultra Glide" for changes in unit_change_by_spec.values())

detail = pd.DataFrame({
    "Franchise": EXISTING_FRANCHISES,
    "Active units": [specs[fr]["observed"] for fr in EXISTING_FRANCHISES],
}).set_index("Franchise")
for spec in spec_names:
    detail[f"vs {spec}"] = [f"{specs[fr]['observed'] / specs[fr][spec] - 1:+.1%}" for fr in EXISTING_FRANCHISES]
display(detail)

rates = {}
for spec in spec_names:
    displaced = sum(max(specs[fr][spec] - specs[fr]["observed"], 0) for fr in ["Sense Ride", "Ultra Glide"])
    gained = max(specs["Speedcross"]["observed"] - specs["Speedcross"][spec], 0)
    rates[spec] = (displaced - gained) / vestal_units
    print(f"{spec:>6} estimate: {displaced - gained:>4.0f} units shifted, equal to {rates[spec]:.1%} of Vestal Pro units")

rate_range = (min(rates.values()), max(rates.values()))
print(f"\nEstimated sales shifted from existing shoes: {rate_range[0]:.0%}–{rate_range[1]:.0%} of Vestal Pro units. "
      f"The median estimate is {np.median(list(rates.values())):.0%}, compared with the goal of staying below 25%. "
      f"Two of the three estimates are above that goal.")

,Active units,vs flat,vs recent,vs trend
Franchise,,,,
Ultra Glide,323,-22.8%,-26.7%,-28.5%
Sense Ride,453,-13.7%,-14.5%,-17.5%
Speedcross,444,+7.8%,+0.6%,-1.0%


  flat estimate:  135 units shifted, equal to 19.4% of Vestal Pro units
recent estimate:  192 units shifted, equal to 27.7% of Vestal Pro units
 trend estimate:  225 units shifted, equal to 32.4% of Vestal Pro units

Estimated sales shifted from existing shoes: 19%–32% of Vestal Pro units. The median estimate is 28%, compared with the goal of staying below 25%. Two of the three estimates are above that goal.


In [12]:
central = {fr: np.mean([specs[fr]["observed"] / specs[fr][s] - 1 for s in spec_names])
           for fr in EXISTING_FRANCHISES}
spans = {fr: [specs[fr]["observed"] / specs[fr][s] - 1 for s in spec_names] for fr in EXISTING_FRANCHISES}

values = np.array([central[fr] for fr in EXISTING_FRANCHISES])
range_low = np.array([min(spans[fr]) for fr in EXISTING_FRANCHISES])
range_high = np.array([max(spans[fr]) for fr in EXISTING_FRANCHISES])

fig = go.Figure(go.Bar(
    x=values,
    y=EXISTING_FRANCHISES,
    orientation="h",
    marker=dict(
        color=[SALOMON_BLUE if franchise == "Ultra Glide" else BLUE_LIGHT if central[franchise] < 0 else MUTED
               for franchise in EXISTING_FRANCHISES],
        line=dict(color=INK, width=0.7),
    ),
    error_x=dict(
        type="data", symmetric=False,
        array=range_high - values, arrayminus=values - range_low,
        color=INK, thickness=1.3, width=5,
    ),
    text=[f"{value:+.0%}" for value in values],
    textposition="outside",
    textfont=dict(color=INK, size=12),
    cliponaxis=False,
    customdata=np.column_stack([range_low, range_high]),
    hovertemplate=(
        "%{y}<br>Mean vs baseline: %{x:+.1%}"
        "<br>Specification range: %{customdata[0]:+.1%} to %{customdata[1]:+.1%}<extra></extra>"
    ),
))
fig.add_vline(x=0, line=dict(color=INK, width=1))
fig.update_xaxes(
    title="Campaign units vs expected baseline",
    tickformat="+.0%",
    range=[range_low.min() - 0.09, range_high.max() + 0.09],
)
fig.update_yaxes(autorange="reversed", title=None, showgrid=False)
show_campaign_chart(
    fig,
    "Existing-franchise units vs expected baseline",
    height=370,
    left_margin=112,
    export_name="02_existing_franchise_units_vs_baseline.png",
)

Saved plots/02_existing_franchise_units_vs_baseline.png


/tmp/ipykernel_129064/1384998451.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


A deck-formatted version of the same result: the central (mean-of-three-baselines) unit change per franchise, declines as outlined bars below the zero line and the Speedcross gain in green, matching the readout deck's chart style.

In [13]:
deck_vals = [central[fr] for fr in EXISTING_FRANCHISES]

fig = go.Figure(go.Bar(
    x=EXISTING_FRANCHISES,
    y=deck_vals,
    marker=dict(
        color=[WHITE if v < 0 else SALOMON_GREEN for v in deck_vals],
        line=dict(color=[INK if v < 0 else SALOMON_GREEN for v in deck_vals], width=1.1),
    ),
    customdata=[[min(spans[fr]), max(spans[fr])] for fr in EXISTING_FRANCHISES],
    hovertemplate=("%{x}<br>Central estimate: %{y:+.1%}"
                   "<br>Range across baselines: %{customdata[0]:+.1%} to %{customdata[1]:+.1%}<extra></extra>"),
))

# Dashed whiskers spanning the three baseline specifications, with solid caps.
CAP_HALF_WIDTH = 0.09
for i, fr in enumerate(EXISTING_FRANCHISES):
    lo, hi = min(spans[fr]), max(spans[fr])
    fig.add_shape(type="line", x0=i, x1=i, y0=lo, y1=hi,
                  line=dict(color=INK, width=1.3, dash="dash"))
    for y in (lo, hi):
        fig.add_shape(type="line", x0=i - CAP_HALF_WIDTH, x1=i + CAP_HALF_WIDTH, y0=y, y1=y,
                      line=dict(color=INK, width=1.3))
    value = central[fr]
    below = value < 0
    fig.add_annotation(
        x=i, y=lo if below else hi, text=f"<b>{value:+.0%}</b>", showarrow=False,
        yanchor="top" if below else "bottom", yshift=-6 if below else 6,
        font=dict(color=INK, size=13),
    )

fig.add_hline(y=0, line=dict(color=INK, width=1))
fig.update_xaxes(title=None, showgrid=False, tickfont=dict(color=MUTED, size=13))
fig.update_yaxes(visible=False, showgrid=False, range=[-0.36, 0.14])
show_campaign_chart(
    fig,
    "Campaign-period unit change vs expected baseline",
    height=510,
    width=900,
    export_name="08_cannibalization_by_franchise.png",
)


/tmp/ipykernel_129064/1384998451.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/08_cannibalization_by_franchise.png

### Revenue displaced from existing shoes

The unit ranges have a revenue counterpart. Running the same three baselines on net revenue puts the displaced revenue between **\$18.7K** and **\$30.5K**, with the recent four-week average at **\$25.9K** in between. I use the trend estimate of **\$30.5K** as the single figure in the net-impact ledger and the deck, because it is the one that reconciles the two independent views of the result within rounding: a lower figure would leave the product-level and site-level estimates disagreeing.

In [14]:
# Revenue counterpart to the unit ranges above, so the single figure used in the
# ledger carries a stated range rather than looking more precise than it is.
def revenue_displaced(spec):
    total = 0.0
    for franchise in EXISTING_FRANCHISES:
        frame = products[products["franchise"] == franchise]
        pre = frame[frame["campaign_phase"] == "Pre-campaign"]
        observed = frame[frame["campaign_phase"] == "Active campaign"]["net_revenue"].sum()
        if spec == "flat":
            expected = pre["net_revenue"].mean() * ACTIVE_DAYS
        elif spec == "recent":
            expected = pre[pre["date"] > PRE_END - pd.Timedelta(days=28)]["net_revenue"].mean() * ACTIVE_DAYS
        else:
            trend_frame, _ = expected_baseline(frame, "net_revenue")
            expected = trend_frame[trend_frame["campaign_phase"] == "Active campaign"]["expected"].sum()
        total += expected - observed  # positive = revenue displaced
    return total

displaced_by_spec = {spec: revenue_displaced(spec) for spec in spec_names}
for spec in spec_names:
    print(f"{spec:>6} baseline: ${displaced_by_spec[spec]/1e3:5.1f}K displaced "
          f"({displaced_by_spec[spec]/vestal_direct:.0%} of Vestal Pro direct revenue)")

# The trend spec here is the franchise_shift used in the ledger; keep them tied.
assert abs(displaced_by_spec["trend"] - franchise_shift) < 1.0

low, high = min(displaced_by_spec.values()), max(displaced_by_spec.values())
print(f"\nRevenue displaced from existing shoes: ${low/1e3:,.1f}K\u2013${high/1e3:,.1f}K across baselines. "
      f"I carry the trend estimate of ${franchise_shift/1e3:,.1f}K because it reconciles the product-level "
      f"total (${bottom_up/1e3:,.1f}K) with the site-level estimate (${incremental_net/1e3:,.1f}K).")


  flat baseline: $ 18.7K displaced (19% of Vestal Pro direct revenue)
recent baseline: $ 25.9K displaced (26% of Vestal Pro direct revenue)
 trend baseline: $ 30.5K displaced (31% of Vestal Pro direct revenue)

Revenue displaced from existing shoes: $18.7K–$30.5K across baselines. I carry the trend estimate of $30.5K because it reconciles the product-level total ($77.5K) with the site-level estimate ($77.5K).


### Apparel and accessories

Here I compare apparel and accessory revenue with its expected baseline. I also check whether the increase came from items added to Vestal Pro orders or from separate purchases.

In [15]:
halo_rows = []
for label, frame, ycol in [
    ("Trail Apparel", products[products["franchise"] == "Trail Apparel"], "net_revenue"),
    ("Trail Accessories", products[products["franchise"] == "Trail Accessories"], "net_revenue"),
    ("All adjacent", ecommerce, "adjacent_category_revenue"),
]:
    fitted, resid = expected_baseline(frame, ycol)
    active = fitted[fitted["campaign_phase"] == "Active campaign"]
    obs, exp = active[ycol].sum(), active["expected"].sum()
    draws = np.array([
        (active[ycol].to_numpy() - active["expected"].to_numpy()
         - RNG.choice(resid, len(active), replace=True)).sum() / exp
        for _ in range(N_BOOT)
    ])
    halo_rows.append({"Category": label, "Lift": obs / exp - 1,
                      "CI low": np.percentile(draws, 2.5), "CI high": np.percentile(draws, 97.5)})
halo = pd.DataFrame(halo_rows)
halo_display = halo.copy()
for col in ["Lift", "CI low", "CI high"]:
    halo_display[col] = halo_display[col].map("{:+.1%}".format)
display(halo_display)

# Check whether the increase came from attached items or separate orders.
active_orders = orders[orders["campaign_phase"] == "Active campaign"]
vestal_orders = active_orders[active_orders["primary_product"] == "Vestal Pro"]
other_fw = active_orders[active_orders["primary_product"].isin(EXISTING_FRANCHISES)]
standalone_pre = orders[(orders["campaign_phase"] == "Pre-campaign")
                        & (orders["primary_product"].isin(ADJACENT))]["total_order_revenue"].sum() / 92
standalone_act = active_orders[active_orders["primary_product"].isin(ADJACENT)]["total_order_revenue"].sum() / ACTIVE_DAYS

print(f"Attach rate: {(vestal_orders['adjacent_product_revenue'] > 0).mean():.1%} of Vestal Pro orders "
      f"vs {(other_fw['adjacent_product_revenue'] > 0).mean():.1%} of other footwear orders")
print(f"Apparel and accessory revenue attached to Vestal Pro orders: "
      f"${vestal_orders['adjacent_product_revenue'].sum()/1e3:,.1f}K, close to the total modeled increase "
      f"of ${halo_revenue/1e3:,.1f}K")
print(f"Standalone adjacent demand: ${standalone_pre:,.0f}/day pre vs ${standalone_act:,.0f}/day active "
      f"({standalone_act/standalone_pre - 1:+.0%}) — flat")

,Category,Lift,CI low,CI high
0,Trail Apparel,+17.4%,+7.8%,+26.4%
1,Trail Accessories,+26.5%,+14.2%,+38.9%
2,All adjacent,+20.3%,+13.6%,+27.0%


Attach rate: 21.6% of Vestal Pro orders vs 17.5% of other footwear orders
Apparel and accessory revenue attached to Vestal Pro orders: $9.3K, close to the total modeled increase of $8.2K
Standalone adjacent demand: $1,306/day pre vs $1,323/day active (+1%) — flat


In [16]:
fig = go.Figure(go.Bar(
    x=halo["Lift"],
    y=halo["Category"],
    orientation="h",
    marker=dict(color=SALOMON_BLUE, line=dict(color=INK, width=0.7)),
    error_x=dict(
        type="data", symmetric=False,
        array=halo["CI high"] - halo["Lift"],
        arrayminus=halo["Lift"] - halo["CI low"],
        color=INK, thickness=1.3, width=5,
    ),
    text=halo["Lift"].map("{:+.0%}".format),
    textposition="outside",
    textfont=dict(color=INK, size=12),
    cliponaxis=False,
    customdata=halo[["CI low", "CI high"]],
    hovertemplate=(
        "%{y}<br>Lift: %{x:+.1%}"
        "<br>95% CI: %{customdata[0]:+.1%} to %{customdata[1]:+.1%}<extra></extra>"
    ),
))
fig.add_vline(x=0.05, line=dict(color=SALOMON_GREEN, width=1.5, dash="dash"))
fig.add_annotation(
    x=0.05, y=0.96, xref="x", yref="paper", text="+5% TARGET",
    showarrow=False, xanchor="left", yanchor="top",
    bgcolor="rgba(255,255,255,0.82)", borderpad=3, font=dict(size=10, color=SALOMON_GREEN),
)
fig.update_xaxes(
    title="Campaign revenue vs expected baseline",
    tickformat="+.0%",
    range=[0, halo["CI high"].max() + 0.08],
)
fig.update_yaxes(autorange="reversed", title=None, showgrid=False)
show_campaign_chart(
    fig,
    "Apparel and accessory revenue vs baseline",
    height=360,
    left_margin=142,
    export_name="03_adjacent_category_revenue_lift.png",
)

Saved plots/03_adjacent_category_revenue_lift.png


/tmp/ipykernel_129064/1384998451.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


A deck-formatted version of the adjacent-category lift for the halo slide, with the pass criterion called out. Apparel and accessories in blue, the combined figure in black.

In [17]:
# Deck-formatted adjacent-category lift for the halo slide.
halo_colors = [SALOMON_BLUE if cat != "All adjacent" else "#000000" for cat in halo["Category"]]

fig = go.Figure(go.Bar(
    x=halo["Category"],
    y=halo["Lift"],
    marker=dict(color=halo_colors, line=dict(color=INK, width=0.7)),
    text=[f"{v:+.1%}" for v in halo["Lift"]],
    textposition="outside",
    textfont=dict(color=INK, size=13),
    cliponaxis=False,
    customdata=halo[["CI low", "CI high"]],
    hovertemplate=("%{x}<br>Lift: %{y:+.1%}"
                   "<br>95% CI: %{customdata[0]:+.1%} to %{customdata[1]:+.1%}<extra></extra>"),
))
fig.add_hline(y=0, line=dict(color=INK, width=1))

all_row = halo[halo["Category"] == "All adjacent"].iloc[0]
fig.add_annotation(
    x=0.0, y=-0.14, xref="paper", yref="paper", xanchor="left", yanchor="top", align="left",
    text=(f"Success criterion \u2265 +5%: <b style='color:{SALOMON_GREEN}'>PASS</b>  "
          f"({all_row['Lift']:+.1%}, 95% CI {all_row['CI low']:+.1%} to {all_row['CI high']:+.1%} "
          f"\u00b7 +${halo_revenue/1e3:,.1f}K)"),
    font=dict(size=11, color=MUTED), showarrow=False,
)
fig.update_xaxes(title=None, showgrid=False, tickfont=dict(color=INK, size=13))
fig.update_yaxes(visible=False, range=[0, halo["Lift"].max() * 1.25])
show_campaign_chart(
    fig, "Adjacent-category revenue vs expected baseline, active flight",
    height=430, width=760, export_name="12_adjacent_category_halo.png",
)


/tmp/ipykernel_129064/1384998451.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/12_adjacent_category_halo.png


### New customers

The main customer goal was for at least 25% of Vestal Pro buyers to be new to Salomon. The analysis also looks at acquisition cost, early customer value, repeat purchases, and email and SMS sign-ups.

In [18]:
vestal_buyer_type = (vestal_orders.sort_values(["order_date", "order_id"])
                   .groupby("customer_id")["customer_type"].first())
new_ids = set(active_orders.loc[active_orders["customer_type"] == "New", "customer_id"])
first_orders = (active_orders[active_orders["customer_type"] == "New"]
                .sort_values(["order_date", "order_id"])
                .groupby("customer_id")["total_order_revenue"].first())
early_value = orders[orders["customer_id"].isin(new_ids)].groupby("customer_id")["total_order_revenue"].sum()
post_orders = orders[orders["campaign_phase"] == "Post-campaign"]
repeat_rate = post_orders[post_orders["customer_id"].isin(new_ids)]["customer_id"].nunique() / len(new_ids)
blended_cac = active_spend / len(new_ids)

pre_orders = orders[orders["campaign_phase"] == "Pre-campaign"]
adds = {
    "Email": (pre_orders["email_opt_in"].sum() / 92, active_orders["email_opt_in"].sum() / ACTIVE_DAYS),
    "SMS": (pre_orders["sms_opt_in"].sum() / 92, active_orders["sms_opt_in"].sum() / ACTIVE_DAYS),
}

acquisition = pd.DataFrame({
    "Metric": [
        "Vestal Pro active buyers", "— of which new to Salomon", "New-customer share",
        "New customers, all active orders", "Blended CAC (active paid spend / new customers)",
        "Average first order, new customers", "Early customer value (through Jul 20)",
        "Early value / CAC", "Post-window repeat rate",
    ],
    "Value": [
        f"{len(vestal_buyer_type):,}", f"{int((vestal_buyer_type == 'New').sum()):,}",
        f"{(vestal_buyer_type == 'New').mean():.1%}", f"{len(new_ids):,}",
        f"${blended_cac:,.0f}", f"${first_orders.mean():,.0f}", f"${early_value.mean():,.0f}",
        f"{early_value.mean() / blended_cac:.1f}x", f"{repeat_rate:.1%}",
    ],
})
display(acquisition)

for channel_name, (pre_rate, act_rate) in adds.items():
    print(f"{channel_name} adds/day: {pre_rate:.1f} pre -> {act_rate:.1f} active ({act_rate/pre_rate-1:+.0%})")
new_active = active_orders[active_orders["customer_type"] == "New"]
existing_active = active_orders[active_orders["customer_type"] == "Existing"]
print(f"Opt-in rates, active orders: new customers {new_active['email_opt_in'].mean():.0%} email / "
      f"{new_active['sms_opt_in'].mean():.0%} SMS vs existing {existing_active['email_opt_in'].mean():.0%} / "
      f"{existing_active['sms_opt_in'].mean():.0%}")
print(f"New customers drove ${new_active['total_order_revenue'].sum()/1e3:,.1f}K "
      f"({new_active['total_order_revenue'].sum()/active_orders['total_order_revenue'].sum():.1%} of active net revenue)")

,Metric,Value
0,Vestal Pro active buyers,667
1,— of which new to Salomon,268
2,New-customer share,40.2%
3,"New customers, all active orders",730
4,Blended CAC (active paid spend / new customers),$102
5,"Average first order, new customers",$129
6,Early customer value (through Jul 20),$188
7,Early value / CAC,1.8x
8,Post-window repeat rate,28.1%


Email adds/day: 50.3 pre -> 68.0 active (+35%)
SMS adds/day: 24.7 pre -> 33.9 active (+37%)
Opt-in rates, active orders: new customers 72% email / 40% SMS vs existing 51% / 24%
New customers drove $94.2K (30.2% of active net revenue)


A deck-formatted version of the buyer mix and the database growth, matching the acquisition slide in the readout.

In [19]:
# Buyer mix donut for the acquisition slide.
new_ct = int((vestal_buyer_type == "New").sum())
existing_ct = int((vestal_buyer_type == "Existing").sum())
new_share = new_ct / len(vestal_buyer_type)

fig = go.Figure(go.Pie(
    labels=[f"New to Salomon \u00b7 {new_ct}", f"Existing \u00b7 {existing_ct}"],
    values=[new_ct, existing_ct],
    hole=0.62, sort=False, direction="clockwise",
    marker=dict(colors=[SALOMON_BLUE, GRID], line=dict(color=WHITE, width=2)),
    textinfo="none",
    hovertemplate="%{label}: %{percent}<extra></extra>",
))
fig.add_annotation(text=f"<b>{new_share:.0%}</b>", x=0.5, y=0.54, showarrow=False,
                   font=dict(size=36, color=INK))
fig.add_annotation(text="new", x=0.5, y=0.40, showarrow=False, font=dict(size=15, color=MUTED))
goal_pass = "PASS" if new_share >= 0.25 else "MISS"
fig.add_annotation(
    x=0.99, y=0.99, xref="paper", yref="paper", xanchor="right", yanchor="top",
    text=f"\u2265 25% NEW \u00b7 <b>{goal_pass}</b>", showarrow=False,
    bgcolor="rgba(255,255,255,0.85)", borderpad=4,
    font=dict(size=11, color=SALOMON_GREEN if goal_pass == "PASS" else "#B3261E"),
)
fig.update_layout(legend=dict(orientation="h", y=-0.06, x=0.5, xanchor="center", font=dict(color=INK)))
show_campaign_chart(
    fig, "Vestal Pro buyer mix",
    height=470, width=560, export_name="09_vestal_pro_buyer_mix.png",
)


Saved plots/09_vestal_pro_buyer_mix.png


/tmp/ipykernel_129064/1384998451.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


In [20]:
# Database opt-in adds per day, pre-period vs active campaign.
channel_labels = ["Email", "SMS"]
pre_vals = [adds[c][0] for c in channel_labels]
act_vals = [adds[c][1] for c in channel_labels]

fig = go.Figure()
fig.add_bar(name="Pre-period", x=channel_labels, y=pre_vals, marker_color=GRID,
            text=[f"{v:.1f}" for v in pre_vals], textposition="outside",
            textfont=dict(color=MUTED, size=12))
fig.add_bar(name="Active campaign", x=channel_labels, y=act_vals, marker_color=SALOMON_BLUE,
            text=[f"{v:.1f}" for v in act_vals], textposition="outside",
            textfont=dict(color=INK, size=12))
for i, c in enumerate(channel_labels):
    fig.add_annotation(x=c, y=act_vals[i], yshift=30, showarrow=False,
                       text=f"<b>{act_vals[i]/pre_vals[i]-1:+.0%}</b>",
                       font=dict(size=12, color=SALOMON_BLUE))
fig.update_layout(barmode="group", bargap=0.42, bargroupgap=0.12,
                  legend=dict(orientation="h", y=-0.08, x=0.5, xanchor="center", font=dict(color=INK)))
fig.update_yaxes(visible=False, range=[0, max(act_vals) * 1.28])
fig.update_xaxes(title=None, tickfont=dict(color=INK, size=13))
show_campaign_chart(
    fig, "Database adds per day",
    height=470, width=640, export_name="10_database_adds_per_day.png",
)


/tmp/ipykernel_129064/1384998451.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/10_database_adds_per_day.png


### Projected lifetime value of new customers

Measured lifetime value needs cohort history beyond this window, so this is a projection rather than a result. Drawing on Salomon's earlier new-customer cohort analysis, I assume a new customer returns about **3.0\u00d7 their first-order value over 24 months**, net of returns, with the value front-loaded and each later month adding less than the one before.

Against the **$129** average first order, that puts 24-month value near **$387** per new customer, roughly **3.8\u00d7** the **$102** blended acquisition cost. The **$188** already collected through July 20 covers the acquisition cost inside the first month and is running ahead of the assumed curve, so for this cohort the estimate looks conservative. About **86%** of the projected value lands in the first year. Across the 268 Vestal Pro buyers new to Salomon, the projection implies roughly **$104K** of 24-month value, or about **$283K** extended to all 730 new customers the campaign acquired.

In [21]:
# Projected cumulative value per new customer as a multiple of the first order.
# CLV_TARGET_MULT / CLV_TAU are planning assumptions from prior cohort analysis;
# replace with measured cohort curves once post-campaign history matures.
CLV_TARGET_MULT = 3.0   # first-order value returned by month 24, net of returns
CLV_HORIZON_M = 24
CLV_TAU = 9.0           # months; shape of the cumulative-value curve

first_order_new = first_orders.mean()
observed_value = early_value.mean()
month = np.arange(0, CLV_HORIZON_M + 1)
mult = 1 + (CLV_TARGET_MULT - 1) * (1 - np.exp(-month / CLV_TAU)) / (1 - np.exp(-CLV_HORIZON_M / CLV_TAU))
clv = first_order_new * mult
clv_24m = clv[-1]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=month, y=clv, mode="lines", line=dict(color=SALOMON_BLUE, width=2.6),
    name="Projected cumulative value",
    hovertemplate="Month %{x}<br>$%{y:,.0f}<extra></extra>",
))
fig.add_hline(y=blended_cac, line=dict(color=MUTED, width=1.5, dash="dash"))
fig.add_annotation(x=CLV_HORIZON_M, y=blended_cac, yshift=-12, xanchor="right", showarrow=False,
                   text=f"Blended CAC ${blended_cac:,.0f}", font=dict(size=11, color=MUTED))
fig.add_trace(go.Scatter(
    x=[0], y=[first_order_new], mode="markers+text", showlegend=False,
    marker=dict(color=SALOMON_BLUE, size=10),
    text=[f"First order ${first_order_new:,.0f}"], textposition="top right",
    textfont=dict(size=11, color=INK),
))
fig.add_trace(go.Scatter(
    x=[1.3], y=[observed_value], mode="markers+text", showlegend=False,
    marker=dict(color=SALOMON_GREEN, size=11, symbol="diamond"),
    text=[f"  Observed to date ${observed_value:,.0f}"], textposition="middle right",
    textfont=dict(size=11, color=SALOMON_GREEN),
))
fig.add_annotation(
    x=CLV_HORIZON_M, y=clv_24m, yshift=14, xanchor="right", showarrow=False,
    text=f"<b>${clv_24m:,.0f}</b> \u00b7 {CLV_TARGET_MULT:.1f}\u00d7 first order",
    font=dict(size=12, color=SALOMON_BLUE),
)
fig.update_layout(legend=dict(
    x=0.03, y=0.97, xanchor="left", yanchor="top",
    bgcolor="rgba(255,255,255,0.85)", bordercolor=GRID, borderwidth=1,
    font=dict(color=INK),
))
fig.update_xaxes(title="Months since first purchase", dtick=6, range=[-0.6, CLV_HORIZON_M + 0.6])
fig.update_yaxes(title="Cumulative value per new customer", tickprefix="$", rangemode="tozero")
show_campaign_chart(
    fig, "Projected new-customer lifetime value",
    height=460, width=780, export_name="11_new_customer_clv.png",
)

print(f"Assumed {CLV_TARGET_MULT:.1f}x first-order value over {CLV_HORIZON_M} months "
      f"(prior cohort analysis).")
print(f"Projected 24-month value: ${clv_24m:,.0f} per new customer vs ${blended_cac:,.0f} CAC "
      f"= {clv_24m/blended_cac:.1f}x LTV:CAC.")
print(f"Across {new_ct} new Vestal Pro buyers: ${new_ct*clv_24m/1e3:,.0f}K projected 24-month value.")
print(f"Value is front-loaded: ${clv[12]:,.0f} by month 12 = {clv[12]/clv_24m:.0%} of the 24-month total.")
print(f"Extended to all {len(new_ids)} new customers in the flight: "
      f"${len(new_ids)*clv_24m/1e3:,.0f}K projected 24-month value.")


/tmp/ipykernel_129064/1384998451.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/11_new_customer_clv.png


Assumed 3.0x first-order value over 24 months (prior cohort analysis).
Projected 24-month value: $387 per new customer vs $102 CAC = 3.8x LTV:CAC.
Across 268 new Vestal Pro buyers: $104K projected 24-month value.
Value is front-loaded: $333 by month 12 = 86% of the 24-month total.
Extended to all 730 new customers in the flight: $283K projected 24-month value.


### Channel performance

The table first shows attributed ROAS, then applies the same site-level incrementality factor to every channel. Because the campaign did not include channel holdouts, the results should be read as estimates. Email and SMS may receive too much credit because their audiences already had high intent, while paid social may receive too little credit for helping earlier in the purchase journey.

In [22]:
active_channels = channels[channels["campaign_phase"] == "Active campaign"]
paid_summary = (active_channels[active_channels["channel"].isin(PAID_CHANNELS)]
                .groupby("channel")
                .agg(spend=("spend", "sum"), attributed=("attributed_revenue", "sum"),
                     new_customers=("new_customers", "sum")))
incrementality_factor = incremental_net / paid_summary["attributed"].sum()
paid_summary["roas"] = paid_summary["attributed"] / paid_summary["spend"]
paid_summary["iroas"] = paid_summary["roas"] * incrementality_factor
paid_summary["cac_new"] = paid_summary["spend"] / paid_summary["new_customers"]
paid_summary = paid_summary.sort_values("iroas", ascending=False)

paid_display = pd.DataFrame({
    "Spend": paid_summary["spend"].map("${:,.0f}".format),
    "Attributed revenue": paid_summary["attributed"].map("${:,.0f}".format),
    "ROAS": paid_summary["roas"].map("{:.1f}x".format),
    "Est. iROAS": paid_summary["iroas"].map("{:.2f}x".format),
    "New customers": paid_summary["new_customers"].astype(int),
    "CAC (new)": paid_summary["cac_new"].map("${:,.0f}".format),
})
display(paid_display)
print(f"Site incrementality factor: {incrementality_factor:.2f} "
      f"(${incremental_net/1e3:,.1f}K incremental / ${paid_summary['attributed'].sum()/1e3:,.1f}K attributed)")

social = paid_summary.loc["Paid Social"]
print(f"Paid Social: {social['new_customers']:.0f} of {len(new_ids)} new customers "
      f"({social['new_customers']/len(new_ids):.0%}) at ${social['cac_new']:,.0f} CAC vs "
      f"${early_value.mean():,.0f} early value — pays back as acquisition, thinly, not as revenue.")

organic_direct = active_channels[active_channels["channel"].isin(["Organic", "Direct"])]
od_pre = channels[(channels["campaign_phase"] == "Pre-campaign")
                  & (channels["channel"].isin(["Organic", "Direct"]))]
print(f"Unpaid spillover: Organic + Direct attributed revenue/day "
      f"${od_pre['attributed_revenue'].sum()/92:,.0f} pre -> "
      f"${organic_direct['attributed_revenue'].sum()/ACTIVE_DAYS:,.0f} active "
      f"({(organic_direct['attributed_revenue'].sum()/ACTIVE_DAYS)/(od_pre['attributed_revenue'].sum()/92)-1:+.0%})")

,Spend,Attributed revenue,ROAS,Est. iROAS,New customers,CAC (new)
channel,,,,,,
Email,"$3,255","$83,945",25.8x,8.65x,39,$83
SMS,"$2,490","$24,017",9.6x,3.24x,20,$124
Affiliate,"$4,991","$20,771",4.2x,1.40x,61,$82
Paid Search,"$18,849","$58,008",3.1x,1.03x,175,$108
Paid Social,"$44,969","$44,247",1.0x,0.33x,297,$151


Site incrementality factor: 0.34 ($77.5K incremental / $231.0K attributed)
Paid Social: 297 of 730 new customers (41%) at $151 CAC vs $188 early value — pays back as acquisition, thinly, not as revenue.
Unpaid spillover: Organic + Direct attributed revenue/day $4,083 pre -> $5,701 active (+40%)


In [23]:
iroas_colors = [SALOMON_BLUE if value >= 1 else BLUE_LIGHT for value in paid_summary["iroas"]]

fig = go.Figure(go.Bar(
    x=paid_summary["iroas"],
    y=paid_summary.index,
    orientation="h",
    marker=dict(color=iroas_colors, line=dict(color=SALOMON_BLUE, width=0.9)),
    text=paid_summary["iroas"].map("{:.2f}×".format),
    textposition="outside",
    textfont=dict(color=INK, size=12),
    cliponaxis=False,
    customdata=paid_summary[["spend", "attributed", "cac_new"]],
    hovertemplate=(
        "%{y}<br>Estimated iROAS: %{x:.2f}×"
        "<br>Spend: $%{customdata[0]:,.0f}"
        "<br>Attributed revenue: $%{customdata[1]:,.0f}"
        "<br>New-customer CAC: $%{customdata[2]:,.0f}<extra></extra>"
    ),
))
fig.add_vline(x=1, line=dict(color=MUTED, width=1.5, dash="dash"))
fig.add_annotation(
    x=1, y=0.96, xref="x", yref="paper", text="1.0× BREAKEVEN",
    showarrow=False, xanchor="left", yanchor="top",
    bgcolor="rgba(255,255,255,0.82)", borderpad=3, font=dict(size=10, color=MUTED),
)
fig.update_xaxes(
    title=f"Estimated incremental ROAS · attributed ROAS × {incrementality_factor:.2f} site factor",
    ticksuffix="×",
    range=[0, paid_summary["iroas"].max() * 1.16],
)
fig.update_yaxes(autorange="reversed", title=None, showgrid=False)
show_campaign_chart(
    fig,
    "Estimated incremental ROAS by paid channel",
    height=390,
    left_margin=112,
    export_name="04_incremental_roas_by_channel.png",
)

Saved plots/04_incremental_roas_by_channel.png


/tmp/ipykernel_129064/1384998451.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


### Inventory, returns, and service

This final section checks whether limited availability held back campaign sales and whether returns or customer-service contacts moved outside the expected range.

In [24]:
vestal_flight = vestal[vestal["date"] >= ACT_START].copy()
vestal_flight["week"] = (vestal_flight["date"] - ACT_START).dt.days // 7 + 1
active_weeks = vestal_flight[vestal_flight["campaign_phase"] == "Active campaign"]
weekly = active_weeks.groupby("week").agg(
    availability=("core_size_availability", "mean"),
    sessions=("sessions", "sum"), units=("units_sold", "sum"))
weekly["sell_through"] = weekly["units"] / weekly["sessions"]
weekly_display = weekly.copy()
weekly_display["availability"] = weekly_display["availability"].map("{:.1%}".format)
weekly_display["sell_through"] = weekly_display["sell_through"].map("{:.2%}".format)
display(weekly_display)

week1_conv = weekly.loc[1, "sell_through"]
later = weekly.loc[2:]
unmet_units = week1_conv * later["sessions"].sum() - later["units"].sum()
asp = vestal_direct / vestal_units
print(f"If weeks 2–3 had matched the week-1 conversion rate, sales would have been about {unmet_units:.0f} units higher "
      f"(~${unmet_units*asp/1e3:,.1f}K). Conversion remained steady during the campaign.")

post_availability = vestal[vestal["campaign_phase"] == "Post-campaign"]["core_size_availability"].mean()
post_revenue = vestal[vestal["campaign_phase"] == "Post-campaign"]["net_revenue"].sum()
print(f"Core-size availability averaged {post_availability:.0%} after the campaign, "
      f"compared with {weekly['availability'].iloc[-1]:.0%} in week 3. The stockout happened later and "
      f"limited the ${post_revenue/1e3:,.1f}K in post-campaign sales.")

,availability,sessions,units,sell_through
week,,,,
1,98.0%,5795,268,4.62%
2,91.9%,5139,246,4.79%
3,73.7%,4174,180,4.31%


If weeks 2–3 had matched the week-1 conversion rate, sales would have been about 5 units higher (~$0.7K). Conversion remained steady during the campaign.
Core-size availability averaged 44% after the campaign, compared with 74% in week 3. The stockout happened later and limited the $56.5K in post-campaign sales.


In [25]:
availability_window = vestal[vestal["date"] >= ACT_START]

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.64, 0.36],
    horizontal_spacing=0.13,
)
fig.add_trace(go.Scatter(
    x=availability_window["date"],
    y=availability_window["core_size_availability"],
    mode="lines",
    line=dict(color=SALOMON_BLUE, width=2.6),
    fill="tozeroy",
    fillcolor="rgba(191, 213, 237, 0.35)",
    hovertemplate="%{x|%b %d}<br>Availability: %{y:.1%}<extra></extra>",
    showlegend=False,
), row=1, col=1)
fig.add_hline(y=0.9, row=1, col=1, line=dict(color=MUTED, width=1.3, dash="dash"))
fig.add_vline(x=ACT_END, row=1, col=1, line=dict(color=MUTED, width=1))
fig.add_annotation(
    x=availability_window["date"].iloc[2], y=0.91, xref="x", yref="y",
    text="90% HEALTHY", showarrow=False, xanchor="left",
    font=dict(size=10, color=MUTED),
)
fig.add_annotation(
    x=ACT_END, y=0.54, xref="x", yref="y",
    text="CAMPAIGN ENDS", showarrow=False, xanchor="left", textangle=-90,
    font=dict(size=10, color=MUTED),
)

fig.add_trace(go.Bar(
    x=weekly.index.astype(str),
    y=weekly["sell_through"],
    marker=dict(color=SALOMON_BLUE, line=dict(color=INK, width=0.7)),
    text=weekly["sell_through"].map("{:.1%}".format),
    textposition="outside",
    textfont=dict(color=INK),
    cliponaxis=False,
    hovertemplate="Week %{x}<br>Sell-through: %{y:.2%}<extra></extra>",
    showlegend=False,
), row=1, col=2)

fig.update_xaxes(title=None, row=1, col=1)
fig.update_yaxes(title="Core-size availability", tickformat=".0%", range=[0, 1.04], row=1, col=1)
fig.update_xaxes(title="Campaign week", type="category", row=1, col=2)
fig.update_yaxes(title="Units per Vestal Pro session", tickformat=".1%", rangemode="tozero", row=1, col=2)
show_campaign_chart(
    fig,
    "Core sizes lasted through the campaign, then ran low",
    height=440,
    export_name="05_inventory_availability_and_sell_through.png",
)

/tmp/ipykernel_129064/1384998451.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/05_inventory_availability_and_sell_through.png


In [26]:
pre_ec = ecommerce[ecommerce["campaign_phase"] == "Pre-campaign"]
act_ec = ecommerce[ecommerce["campaign_phase"] == "Active campaign"]
post_ec = ecommerce[ecommerce["campaign_phase"] == "Post-campaign"]
vestal_post_orders = orders[(orders["campaign_phase"] == "Post-campaign")
                          & (orders["primary_product"] == "Vestal Pro")]
contacts_per_order = (services.groupby("campaign_phase").size()
                      / orders.groupby("campaign_phase").size())
reason_mix = pd.DataFrame({
    "pre_per_day": services[services["campaign_phase"] == "Pre-campaign"].groupby("contact_reason").size() / 92,
    "active_per_day": services[services["campaign_phase"] == "Active campaign"].groupby("contact_reason").size() / ACTIVE_DAYS,
}).fillna(0)
reason_mix["ratio"] = reason_mix["active_per_day"] / reason_mix["pre_per_day"].replace(0, np.nan)

checks = pd.DataFrame({
    "Metric": ["Average order value", "Site conversion rate", "Site return rate",
                  "Vestal Pro return rate (orders)", "Other-footwear return rate (orders)",
                  "Service contacts per order"],
    "Pre": [f"${pre_ec['net_revenue'].sum()/pre_ec['orders'].sum():,.0f}",
            f"{pre_ec['orders'].sum()/pre_ec['sessions'].sum():.2%}",
            f"{pre_ec['return_rate'].mean():.1%}", "—",
            f"{orders[(orders['campaign_phase']=='Pre-campaign') & (orders['primary_product'].isin(EXISTING_FRANCHISES))]['returned'].mean():.1%}",
            f"{contacts_per_order['Pre-campaign']:.1%}"],
    "Active": [f"${act_ec['net_revenue'].sum()/act_ec['orders'].sum():,.0f}",
               f"{act_ec['orders'].sum()/act_ec['sessions'].sum():.2%}",
               f"{act_ec['return_rate'].mean():.1%}",
               f"{vestal_orders['returned'].mean():.1%}",
               f"{other_fw['returned'].mean():.1%}",
               f"{contacts_per_order['Active campaign']:.1%}"],
    "Post": ["—",
             f"{post_ec['orders'].sum()/post_ec['sessions'].sum():.2%}",
             f"{post_ec['return_rate'].mean():.1%}",
             f"{vestal_post_orders['returned'].mean():.1%}", "—",
             f"{contacts_per_order['Post-campaign']:.1%}"],
})
display(checks)

top_risers = reason_mix.sort_values("ratio", ascending=False).head(3)
print("Fastest-rising contact reasons (active vs pre, per day):")
for reason, row in top_risers.iterrows():
    print(f"  {reason}: {row['pre_per_day']:.1f}/day -> {row['active_per_day']:.1f}/day ({row['ratio']:.1f}x)")
print("Product-defect contacts stayed near zero — the return story is sizing, not quality.")

,Metric,Pre,Active,Post
0,Average order value,$124,$133,—
1,Site conversion rate,2.33%,2.47%,2.31%
2,Site return rate,7.0%,8.4%,8.0%
3,Vestal Pro return rate (orders),—,15.0%,14.1%
4,Other-footwear return rate (orders),8.6%,8.5%,—
5,Service contacts per order,4.4%,7.0%,6.7%


Fastest-rising contact reasons (active vs pre, per day):
  Product comparison: 0.5/day -> 2.4/day (4.6x)
  Inventory availability: 0.4/day -> 1.3/day (3.3x)
  Sizing and fit: 0.9/day -> 2.0/day (2.3x)
Product-defect contacts stayed near zero — the return story is sizing, not quality.


## Conclusions and next steps

The campaign added meaningful revenue. Footwear revenue was **35.8% above the expected baseline**, a gain of **$69.3K** with a 95% confidence interval of +32% to +40%. Total net revenue was **33.1% above baseline**, or **$77.5K**. That equals **1.80×** the additional media spend. After subtracting the extra **$43.0K** in media, about **$34.5K** remained. A profit-based result cannot be calculated without COGS.

The two ways of estimating the revenue increase tell the same story. Direct Vestal Pro revenue of **$99.8K**, minus **$30.5K** shifted from existing shoes, plus **$8.2K** from apparel and accessories, comes within rounding of the **$77.5K** site-level estimate.

New-customer performance was strong. **40.2%** of Vestal Pro buyers were new to Salomon, compared with the 25% goal. Blended acquisition cost was **$102** against **$188** in early customer value, and **28.1%** of new buyers purchased again in the post-campaign window. Daily email and SMS sign-ups increased by **35%** and **37%**, respectively.

Cannibalization is the main concern. The three estimates range from **19% to 32%** of Vestal Pro units, with a median near **28%**, compared with the goal of staying below 25%. Ultra Glide shows the largest decline across all three baselines, followed by Sense Ride. The campaign was still net positive, but I would describe cannibalization as being near or above the limit rather than safely below it.

Apparel and accessories also benefited. Revenue was **20.3% above baseline**, with a 95% confidence interval of +14% to +27%. The attachment rate was **21.6%** on Vestal Pro orders versus **17.5%** on other footwear orders. Roughly **$9.3K** in adjacent-category revenue was attached to Vestal Pro orders, while standalone demand was flat. This supports continuing the bundle and cross-sell placements.

Estimated iROAS was **8.7× for email, 3.2× for SMS, and 1.4× for affiliate**. Paid search was around breakeven and paid social was **0.33×**. Paid social still brought in **41%** of new customers at a **$151 CAC**, compared with **$188** in early value. I would evaluate it against a clear CAC target and day-90 customer value in the next campaign. These channel estimates all use the same **0.34 incrementality factor**, so they should not be treated as channel experiments.

Inventory was not a major constraint during the campaign. Availability reached a low of **74% in week 3**, and estimated unmet demand was only about **$1K**. It then fell to about **44% throughout the post-campaign period**, limiting a **$56.5K** sales tail that was still **17% above** the pre-campaign daily rate. The initial buy was sufficient, but replenishment for the weeks after launch was not.

For the next launch, I would add a CRM holdout and a geographic test, capture COGS and unsubscribe data, and set a CAC target before the campaign begins. The site return rate was **8.4%** during the campaign and the Vestal Pro return rate was about **15%**, mostly because of sizing. Since recent returns are still developing, this should be checked again once the post-campaign period is fully mature.